#### 06 — Fine-tuning Transformer for Hallucination Detection

In this notebook we fine-tune a transformer encoder on the hallucination
detection task using prompt–response pairs.

Goal:
Evaluate whether task-adapted representation learning can surpass
strong classical baselines (TF-IDF + numeric features).

In [1]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [2]:
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

from src.data.load_splits import load_splits


/Users/aviv.gross/hallu-detect/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
train_df, val_df, test_df = load_splits(ROOT)

print(train_df.shape, val_df.shape, test_df.shape)
train_df[["prompt", "response", "label"]].head(2)


(51605, 6) (6451, 6) (6451, 6)


,prompt,response,label
0,Johnny Mathis Sings included a song that start...,Blake Edwards,0
1,Usain Bolt says Tyson Gay should have been giv...,Usain Bolt and Tyson Gay are set to compete ag...,1


#### Build model input

In [4]:
SEP_TOKEN = " [SEP] "

def build_input_text(df):
    return (df["prompt"].fillna("") + SEP_TOKEN + df["response"].fillna("")).tolist()

train_texts = build_input_text(train_df)
val_texts   = build_input_text(val_df)
test_texts  = build_input_text(test_df)

train_labels = train_df["label"].values
val_labels   = val_df["label"].values
test_labels  = test_df["label"].values


#### Dataset class

In [5]:
class HalluDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.enc = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=max_length,
        )
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


#### Tokenizer & datasets

In [6]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

MAX_LEN = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds = HalluDataset(train_texts, train_labels, tokenizer, max_length=MAX_LEN)
val_ds   = HalluDataset(val_texts,   val_labels,   tokenizer, max_length=MAX_LEN)
test_ds  = HalluDataset(test_texts,  test_labels,  tokenizer, max_length=MAX_LEN)


#### Metrics

In [7]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall,
    }


#### MODEL

In [8]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at sentence-transformers/all-MiniLM-L6-v2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


#### Training arguments

In [9]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="../models/finetuned_transformer",
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,

    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    logging_steps=100,
    report_to="none",

    remove_unused_columns=False,
    dataloader_num_workers=0,
    dataloader_pin_memory=False,   # silence MPS warning

    fp16=False,                    # keep false on Mac/MPS unless you're sure
)



#### Trainer

In [10]:
from transformers import Trainer, EarlyStoppingCallback

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=6)],
)


/var/folders/w9/fltsbn2x0vz7jxpbj2rgyr6c0000gn/T/ipykernel_10710/2596512828.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [11]:
import torch

use_mps = torch.backends.mps.is_available()
device = "mps" if use_mps else "cpu"
print("Using device:", device)


Using device: mps


#### Train

In [12]:
trainer.train()

Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
500,0.418600,0.419499,0.759107,0.771806,0.689764,0.876000
1000,0.405100,0.398980,0.769338,0.695082,0.902128,0.565333
1500,0.419300,0.405600,0.769958,0.709589,0.859242,0.604333
2000,0.401300,0.384466,0.774144,0.787206,0.700546,0.898333
2500,0.405900,0.374480,0.780034,0.792453,0.706020,0.903000
3000,0.381000,0.395478,0.771973,0.786099,0.697189,0.901000
3500,0.371200,0.400209,0.776004,0.724289,0.846943,0.632667
4000,0.369000,0.413237,0.776624,0.795516,0.692612,0.934333
4500,0.351600,0.385726,0.782204,0.795160,0.706660,0.909000
5000,0.391800,0.386313,0.778949,0.721810,0.870179,0.616667


TrainOutput(global_step=9678, training_loss=0.3791809970865961, metrics={'train_runtime': 2991.3804, 'train_samples_per_second': 51.754, 'train_steps_per_second': 3.235, 'total_flos': 2567282411566080.0, 'train_loss': 0.3791809970865961, 'epoch': 3.0})

#### EVAL

In [13]:
val_metrics  = trainer.evaluate(val_ds)
test_metrics = trainer.evaluate(test_ds)

print("Validation:", val_metrics)
print("Test:", test_metrics)

trainer.save_model("../models/finetuned_transformer/best_model")

Validation: {'eval_loss': 0.40752550959587097, 'eval_accuracy': 0.7794140443342118, 'eval_f1': 0.7993796700972791, 'eval_precision': 0.6926459809430735, 'eval_recall': 0.945, 'eval_runtime': 24.6207, 'eval_samples_per_second': 262.015, 'eval_steps_per_second': 8.204, 'epoch': 3.0}
Test: {'eval_loss': 0.41268113255500793, 'eval_accuracy': 0.7766237792590296, 'eval_f1': 0.7975270479134466, 'eval_precision': 0.6893368957979111, 'eval_recall': 0.946, 'eval_runtime': 24.4636, 'eval_samples_per_second': 263.698, 'eval_steps_per_second': 8.257, 'epoch': 3.0}


## Conclusions — Fine-tuned Transformer Models

In this notebook, we evaluated whether end-to-end fine-tuning of a transformer-based
model can outperform strong classical baselines for hallucination detection.

We fine-tuned a MiniLM-based encoder on prompt–response pairs using a binary
classification head, training for three full epochs with early stopping and
multiple evaluation checkpoints. We experimented with longer input sequences
(max_length=256) to ensure sufficient context coverage.

Despite stable training and extensive fine-tuning, the transformer model did not
surpass the classical TF-IDF(response)+numeric baseline.

### Key findings

- The fine-tuned transformer achieved a **test F1 score of ~0.80** and
  **accuracy of ~0.78**, compared to **~0.82 accuracy and ~0.82 F1** for the
  TF-IDF(response)+numeric baseline.
- The transformer consistently exhibited **very high recall (~0.95)** but
  **low precision (~0.69)**, indicating a strong tendency to over-predict
  hallucinations.
- Increasing the input length and extending training did not materially improve
  performance, suggesting the model reached a performance plateau on this dataset.

### Interpretation

These results suggest that hallucination signals in the HaluEval dataset are
largely captured by **surface-level textual cues**—such as phrasing patterns,
hedging language, and stylistic markers—which are effectively modeled by
TF-IDF representations combined with simple numeric features.

While the transformer model is capable of learning broad semantic patterns,
it appears less effective at distinguishing between *cautious but correct*
responses and genuine hallucinations, leading to reduced precision.

### Implications

The findings indicate that, for this dataset, **classical feature-based approaches
remain highly competitive**, and in fact outperform neural models that rely
primarily on semantic representations.

This motivates a deeper investigation into model robustness and generalization:
a transformer model may still provide advantages when evaluated on datasets
with different hallucination characteristics or reduced reliance on dataset-specific
surface cues.

Accordingly, the next stage of this work focuses on **cross-dataset generalization**
to assess whether neural models offer improved robustness beyond the HaluEval setting.
